# sidecar_rs: reading and writing SCAR sidecar files

This notebook demonstrates the `sidecar_rs` Python bindings end to end: creating a
sidecar document, writing it (to a file and to bytes), reading it back, inspecting it,
and handling errors.

A SCAR sidecar is a compact binary file that stores a typed key-value catalog of
metadata alongside a media file (for example `photo.jpg` -> `photo.scar`).

> **Setup:** install the package from the repo root first, e.g. `uv pip install .`
> (or `make py-build`), then run this notebook with that environment's kernel.

In [ ]:
import sidecar_rs
from sidecar_rs import SidecarDocument, SidecarError, conventions

print("sidecar extension:", sidecar_rs.SIDECAR_EXTENSION)

## 1. Writing a sidecar

Create a document and set fields. Values can be set generically (the type is inferred)
or with explicit typed setters when precision matters (e.g. `set_f32`). Photography
keys come from `sidecar_rs.conventions`.

In [ ]:
doc = SidecarDocument()

# Inferred types: float -> F64, int -> I64/U64, str -> String, list -> Array
doc.set(conventions.GPS_LATITUDE, 37.7749)
doc.set(conventions.GPS_LONGITUDE, -122.4194)
doc.set(conventions.EXPOSURE_TIME_SEC, 0.008)
doc.set(conventions.EXPOSURE_ISO, 400)
doc.set(conventions.RATING, 4)
doc.set(conventions.TAGS, ["spring", "outdoor", "macro"])

# Explicit 32-bit float to avoid widening to f64
doc.set_f32(conventions.LENS_FOCAL_LENGTH_MM, 50.0)

# Record which media file this sidecar belongs to
doc.set_media_basename("photo.jpg")

print(doc)
print("keys:", doc.keys())

## 2. Persisting: to a file and to bytes

`to_path` writes a `.scar` file; `to_bytes` returns the raw binary, handy for storing
in a database or sending over a network.

In [ ]:
from pathlib import Path

sidecar_path = Path("photo.scar")
doc.to_path(sidecar_path)

data = doc.to_bytes()
print("serialized", len(data), "bytes")
print("magic:", data[:4])
print("header hex:", data[:16].hex(" "))

## 3. Reading it back

Load from the file and from the in-memory bytes, and confirm the round-trip preserved
the values (including the exact `f64` GPS coordinate).

In [ ]:
from_file = SidecarDocument.from_path(sidecar_path)
from_mem = SidecarDocument.from_bytes(data)

assert from_file[conventions.GPS_LATITUDE] == 37.7749
assert from_mem[conventions.TAGS] == ["spring", "outdoor", "macro"]
assert from_mem.media_basename() == "photo.jpg"

print("latitude:", from_file[conventions.GPS_LATITUDE])
print("tags:", from_mem[conventions.TAGS])
print("round-trip OK")

## 4. Inspecting a document

`SidecarDocument` behaves like a mapping: `len()`, `in`, indexing, and `del` all work,
alongside `entries()`, `keys()`, and `header()`.

In [ ]:
print("entry count:", len(from_mem))
print("has rating:", conventions.RATING in from_mem)
print("header:", from_mem.header())
print()
for key, value in from_mem.entries().items():
    print(f"  {key} = {value!r}")

# Dict-style mutation
from_mem["photo.caption"] = "Cherry blossoms"
del from_mem[conventions.RATING]
print()
print("after edit:", sorted(from_mem.keys()))

## 5. Error handling

Invalid data raises `sidecar_rs.SidecarError`.

In [ ]:
try:
    SidecarDocument.from_bytes(b"not a scar file")
except SidecarError as exc:
    print("caught SidecarError:", exc)

## 6. Cleanup

Remove the temporary `.scar` file written above.

In [ ]:
sidecar_path.unlink(missing_ok=True)
print("removed", sidecar_path)